In [ ]:
import spacy
from collections import Counter


MODEL_NAME = "en_core_web_sm"
TEST_FILE = "test.txt"


def load_test_data(filename):
    """
    讀取 test.txt

    格式：
    text<TAB>entity|label;entity|label
    """

    data = []

    with open(filename, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            # 跳過空行
            if not line:
                continue

            # 必須有 TAB
            if "\t" not in line:
                print(f"[警告] 第 {line_number} 行格式錯誤，缺少 TAB")
                continue

            text, annotation = line.split("\t", 1)

            expected_entities = []

            if annotation.strip():
                for item in annotation.split(";"):
                    item = item.strip()

                    if not item:
                        continue

                    if "|" not in item:
                        print(
                            f"[警告] 第 {line_number} 行 Entity 格式錯誤：{item}"
                        )
                        continue

                    entity_text, label = item.rsplit("|", 1)

                    expected_entities.append(
                        {
                            "text": entity_text,
                            "label": label
                        }
                    )

            data.append(
                {
                    "line": line_number,
                    "text": text,
                    "expected": expected_entities
                }
            )

    return data


def find_entity_spans(text, entities):
    """
    將 Entity 文字轉成：

    {
        "start": xxx,
        "end": xxx,
        "label": "ORG",
        "text": "Apple"
    }

    使用 find 找到 Entity 在原文中的位置。
    """

    result = []

    used_positions = set()

    for entity in entities:
        entity_text = entity["text"]
        label = entity["label"]

        start = 0

        while True:
            pos = text.find(entity_text, start)

            if pos == -1:
                break

            end = pos + len(entity_text)

            # 避免同一位置被重複使用
            if (pos, end) not in used_positions:
                result.append(
                    {
                        "start": pos,
                        "end": end,
                        "label": label,
                        "text": entity_text
                    }
                )

                used_positions.add((pos, end))
                break

            start = pos + 1

    return result


def entity_key(entity):
    """
    Entity 比較用 Key

    例如：

    Apple / ORG

    會變成：

    (0, 5, 'ORG')
    """

    return (
        entity["start"],
        entity["end"],
        entity["label"]
    )


def evaluate(nlp, test_data):
    """
    執行 NER 評估
    """

    total_expected = 0
    total_predicted = 0
    total_correct = 0

    label_expected = Counter()
    label_predicted = Counter()
    label_correct = Counter()

    errors = []

    print("=" * 80)
    print("NER Test")
    print("=" * 80)

    for index, item in enumerate(test_data, start=1):

        text = item["text"]

        # 模型辨識
        doc = nlp(text)

        predicted_entities = []

        for ent in doc.ents:
            predicted_entities.append(
                {
                    "start": ent.start_char,
                    "end": ent.end_char,
                    "label": ent.label_,
                    "text": ent.text
                }
            )

        # 預期答案轉成 span
        expected_entities = find_entity_spans(
            text,
            item["expected"]
        )

        expected_keys = set(
            entity_key(e)
            for e in expected_entities
        )

        predicted_keys = set(
            entity_key(e)
            for e in predicted_entities
        )

        correct_keys = expected_keys & predicted_keys

        total_expected += len(expected_keys)
        total_predicted += len(predicted_keys)
        total_correct += len(correct_keys)

        # Label 統計
        for entity in expected_entities:
            label_expected[entity["label"]] += 1

        for entity in predicted_entities:
            label_predicted[entity["label"]] += 1

        for entity in expected_entities:
            if entity_key(entity) in correct_keys:
                label_correct[entity["label"]] += 1

        # 判斷整句是否完全正確
        sentence_correct = (
            expected_keys == predicted_keys
        )

        print()
        print(f"[{index}] {text}")

        print("  預期：", end=" ")

        if expected_entities:
            print(
                ", ".join(
                    f"{e['text']} [{e['label']}]"
                    for e in expected_entities
                )
            )
        else:
            print("無")

        print("  模型：", end=" ")

        if predicted_entities:
            print(
                ", ".join(
                    f"{e['text']} [{e['label']}]"
                    for e in predicted_entities
                )
            )
        else:
            print("無")

        if sentence_correct:
            print("  結果：✓ 正確")
        else:
            print("  結果：✗ 錯誤")

            errors.append(
                {
                    "line": item["line"],
                    "text": text,
                    "expected": expected_entities,
                    "predicted": predicted_entities
                }
            )

    # Precision / Recall / F1
    precision = (
        total_correct / total_predicted
        if total_predicted > 0
        else 0
    )

    recall = (
        total_correct / total_expected
        if total_expected > 0
        else 0
    )

    if precision + recall > 0:
        f1 = (
            2 * precision * recall
            / (precision + recall)
        )
    else:
        f1 = 0

    print()
    print("=" * 80)
    print("總結果")
    print("=" * 80)

    print(f"預期 Entity 數量 : {total_expected}")
    print(f"模型 Entity 數量 : {total_predicted}")
    print(f"正確 Entity 數量 : {total_correct}")
    print(f"錯誤案例數量     : {len(errors)}")

    print()
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    # Entity-level Accuracy
    if total_expected > 0:
        entity_accuracy = total_correct / total_expected
    else:
        entity_accuracy = 0

    print(f"Entity Accuracy : {entity_accuracy:.4f}")

    # 每種 Entity Label
    print()
    print("=" * 80)
    print("各 Entity 類型結果")
    print("=" * 80)

    all_labels = sorted(
        set(label_expected) |
        set(label_predicted)
    )

    print(
        f"{'Label':<12}"
        f"{'Expected':<12}"
        f"{'Predicted':<12}"
        f"{'Correct':<12}"
        f"{'Precision':<12}"
        f"{'Recall':<12}"
        f"{'F1':<12}"
    )

    for label in all_labels:

        expected = label_expected[label]
        predicted = label_predicted[label]
        correct = label_correct[label]

        p = correct / predicted if predicted else 0
        r = correct / expected if expected else 0

        if p + r > 0:
            label_f1 = 2 * p * r / (p + r)
        else:
            label_f1 = 0

        print(
            f"{label:<12}"
            f"{expected:<12}"
            f"{predicted:<12}"
            f"{correct:<12}"
            f"{p:<12.4f}"
            f"{r:<12.4f}"
            f"{label_f1:<12.4f}"
        )

    # 顯示錯誤案例
    print()
    print("=" * 80)
    print("錯誤案例")
    print("=" * 80)

    if not errors:
        print("沒有錯誤案例！")
    else:

        for error in errors:

            print()
            print(f"第 {error['line']} 行")
            print(f"Text: {error['text']}")

            print("Expected:")

            if error["expected"]:
                for e in error["expected"]:
                    print(
                        f"  - {e['text']} [{e['label']}]"
                    )
            else:
                print("  - 無")

            print("Predicted:")

            if error["predicted"]:
                for e in error["predicted"]:
                    print(
                        f"  - {e['text']} [{e['label']}]"
                    )
            else:
                print("  - 無")

            print("-" * 50)


def main():

    print("載入模型...")

    try:
        nlp = spacy.load(MODEL_NAME)
    except OSError:
        print()
        print(f"找不到模型：{MODEL_NAME}")
        print()
        print("請先執行：")
        print("python -m spacy download en_core_web_sm")
        return

    print(f"模型：{MODEL_NAME}")

    # 載入 Test Dataset
    test_data = load_test_data(TEST_FILE)

    if not test_data:
        print("test.txt 沒有有效的測試資料")
        return

    print(f"測試資料：{len(test_data)} 筆")

    # 開始測試
    evaluate(nlp, test_data)


if __name__ == "__main__":
    main()

載入模型...


d:\2.programm2\github-star\presidio-research\venvpr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


模型：en_core_web_sm
測試資料：10 筆
NER Test

[1] Apple is looking at buying a startup in London.
  預期： Apple [ORG], London [GPE]
  模型： Apple [ORG], London [GPE]
  結果：✓ 正確

[2] Barack Obama was born in Hawaii.
  預期： Barack Obama [PERSON], Hawaii [GPE]
  模型： Barack Obama [PERSON], Hawaii [GPE]
  結果：✓ 正確

[3] Microsoft was founded by Bill Gates.
  預期： Microsoft [ORG], Bill Gates [PERSON]
  模型： Microsoft [ORG], Bill Gates [PERSON]
  結果：✓ 正確

[4] Google has its headquarters in Mountain View, California.
  預期： Google [ORG], Mountain View [GPE], California [GPE]
  模型： Google [ORG], Mountain View [GPE], California [GPE]
  結果：✓ 正確

[5] Elon Musk is the CEO of Tesla.
  預期： Elon Musk [PERSON], Tesla [ORG]
  模型： Elon Musk [PERSON], Tesla [ORG]
  結果：✓ 正確

[6] The Eiffel Tower is located in Paris, France.
  預期： Eiffel Tower [FAC], Paris [GPE], France [GPE]
  模型： The Eiffel Tower [LOC], Paris [GPE], France [GPE]
  結果：✗ 錯誤

[7] Amazon was founded in Seattle.
  預期： Amazon [ORG], Seattle [GPE]
  模型： Amazon [OR

In [ ]:
import spacy


MODEL_NAME = "en_core_web_sm"
TEST_FILE = "test.txt"


# 你認為哪些 Entity 類型屬於敏感資訊
SENSITIVE_LABELS = {
    "PERSON",   # 人名
    "GPE",       # 國家、城市、地區
    "LOC",       # 地點
    "ORG",       # 組織
    "DATE",      # 日期
    "TIME",      # 時間
    "MONEY",     # 金額
    "CARDINAL",  # 數字
    "FAC",       # 設施
}


def load_test_data(filename):
    """
    test.txt 格式：

    句子<TAB>YES
    句子<TAB>NO
    """

    data = []

    with open(filename, "r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            if "\t" not in line:
                print(f"[警告] 第 {line_number} 行格式錯誤")
                continue

            text, expected = line.split("\t", 1)

            expected = expected.strip().upper()

            if expected not in {"YES", "NO"}:
                print(
                    f"[警告] 第 {line_number} 行答案必須是 YES 或 NO"
                )
                continue

            data.append({
                "line": line_number,
                "text": text,
                "expected": expected
            })

    return data


def detect_sensitive(nlp, text):
    """
    使用 en_core_web_sm 辨識 Entity。

    只要出現 SENSITIVE_LABELS 中的 Entity，
    就判定這句話有敏感資訊。
    """

    doc = nlp(text)

    sensitive_entities = []

    for ent in doc.ents:

        if ent.label_ in SENSITIVE_LABELS:

            sensitive_entities.append({
                "text": ent.text,
                "label": ent.label_
            })

    if sensitive_entities:
        return "YES", sensitive_entities

    return "NO", []


def main():

    print("Loading model...")

    try:
        nlp = spacy.load(MODEL_NAME)

    except OSError:

        print()
        print("找不到 en_core_web_sm")
        print()
        print("請先執行：")
        print("python -m spacy download en_core_web_sm")
        return

    test_data = load_test_data(TEST_FILE)

    if not test_data:
        print("test.txt 沒有測試資料")
        return

    correct = 0
    total = len(test_data)

    print()
    print("=" * 80)
    print("NER Sensitive Information Test")
    print("=" * 80)

    for index, item in enumerate(test_data, start=1):

        text = item["text"]
        expected = item["expected"]

        predicted, entities = detect_sensitive(
            nlp,
            text
        )

        is_correct = predicted == expected

        if is_correct:
            correct += 1

        print()
        print(f"[{index}] {text}")
        print(f"Expected : {expected}")
        print(f"Predicted: {predicted}")

        if entities:
            entity_text = ", ".join(
                f"{e['text']} [{e['label']}]"
                for e in entities
            )

            print(f"Detected : {entity_text}")

        if is_correct:
            print("Result   : CORRECT")
        else:
            print("Result   : WRONG")

    accuracy = correct / total

    print()
    print("=" * 80)
    print("FINAL RESULT")
    print("=" * 80)

    print(f"Total   : {total}")
    print(f"Correct : {correct}")
    print(f"Wrong   : {total - correct}")
    print(f"Accuracy: {accuracy:.2%}")

    print("=" * 80)


if __name__ == "__main__":
    main()

Loading model...

NER Sensitive Information Test

[1] John Smith recently moved from Boston to New York because he accepted a new position at Microsoft and will start working at the company's Manhattan office next month.
Expected : YES
Predicted: YES
Detected : John Smith [PERSON], Boston [GPE], New York [GPE], Microsoft [ORG], Manhattan [GPE], next month [DATE]
Result   : CORRECT

[2] I spent most of the weekend cleaning my apartment, watching movies, ordering food, and reading a book that I had bought several months ago.
Expected : NO
Predicted: YES
Detected : the weekend [DATE], several months ago [DATE]
Result   : WRONG

[3] Sarah told her manager that she would be attending a conference in London next Tuesday and would probably stay near the airport for three nights.
Expected : YES
Predicted: YES
Detected : Sarah [PERSON], London [GPE], next Tuesday [DATE], three nights [DATE]
Result   : CORRECT

[4] The new restaurant opened last weekend and received several positive reviews from